# Introduction

The goals of this analysis is to train, optimize, predict, evaluate, and SHAP BPNet models trained on Drosophila and Cnidarian HOX proteins. 

# Computational setup

In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import warnings
warnings.filterwarnings("ignore")
from tensorflow.python.util import deprecation
deprecation._PRINT_DEPRECATION_WARNINGS = False

#Packages
import os
import sys
import keras
import pyBigWig
import json
import pandas as pd
import numpy as np
import keras.backend as K
from keras.models import load_model
from tqdm import tqdm

import plotnine
from plotnine import *

# Settings

## Working options
os.chdir(f'/n/projects/mw2098/collaboration/for_carlos/20260205_manuscript/2_modeling/')
pd.set_option('display.max_columns', 100)
figure_path = 'figures/3_train_solo'
bpreveal_path = '/home/mw2098/bin/bpreveal_510/'
python_path = '/home/mw2098/anaconda3/envs/bpreveal_510/bin/python'

## Custom functions
sys.path.insert(0, f'scripts/py/functions/')
from metrics import compute_auprc
from functional import one_hot_encode_sequences, one_hot_encode_sequence, one_hot_decode_sequence, shuffle_seqs, logitsToProfile, insert_motif
from perturb import generate_random_seq
from motifs import extract_seqs_from_df, resize_coordinates

sys.path.insert(0, f'{bpreveal_path}/src')
import losses

## Filesystem commands
!mkdir -p \
    json/optimize \
    bed/bpreveal/optimize \
    input/optimize \
    models/optimize \
    preds/optimize \
    scripts/modeling/optimize \
    tsv/optimize \
    tsv/insilico \
    shap \
    npz \
    {figure_path}

2026-03-05 08:53:28.392811: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772722408.411931 4031351 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772722408.417808 4031351 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-05 08:53:28.438852: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-05 08:53:34.884437: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: 

## Custom functions

The following function takes a list of str commands and generates a SLURM batch job with parameter allocations. This is mostly specific to Stowers computational resources, would recommend reworking the job submissions to match your own institutional setup if reproducing.

In [2]:
def generate_slurm_array(cmds, output_file, mem = 200, simultaneous_jobs = 3, gpu = True):
    total_jobs = len(cmds)
    if gpu:
        # gpu_fill = '#SBATCH --partition=gpu\n#SBATCH --gres=gpu:a100:1\n#SBATCH --cpus-per-task=10'
        gpu_fill = '#SBATCH --partition=gpu\n#SBATCH --gres=gpu:1\n#SBATCH --cpus-per-task=10'
    else:
        gpu_fill = '#SBATCH --cpus-per-task=10'
    array_header = ['#!/usr/bin/bash',
                  '#SBATCH --job-name bpnet_training',
                  '#SBATCH --output=slurm_%j.log',
                  f'#SBATCH --array=1-{total_jobs}%{simultaneous_jobs}',
                  f'#SBATCH --mem={mem}gb', 
                  '#SBATCH --time=72:00:00',
                  gpu_fill,
                  'source /home/mw2098/.bashrc',
                  'conda deactivate',
                  'conda activate bpreveal_510',
                  'ml bedtools',
                  f'cd {working_dir}']
    # array_cmds = array_header
    for i,cmd in enumerate(cmds):
        array_cmd = ["if [[ ${{SLURM_ARRAY_TASK_ID}} == {0:d} ]] ; then\n".format(i+1), 
                    "    {0:s}\n".format(cmd),
                    "fi\n\n"]
        array_header = array_header + array_cmd 
        
    with open(output_file, mode='wt') as slurm:
        slurm.write('\n'.join(array_header))
        slurm.write('\n')

# Set up static modeling parameters

Here are the variables that will remain static throughout the optimization process. 

In [3]:
# Default model architecture
batch_size = 16
learning_rate = 0.004 #BPReveal is adaptive
early_stopping_patience = 11
reverse_complement = True
seed = 2346
trials = 64

#Annotations
working_dir = os.getcwd()
genome = '../public/databases/UCSC/dm6.fa'
test_chroms = ['chrX']
val_chroms = ['chr2R']
train_chroms = ['chr3L', 'chr3R', 'chr2L']
profile_metrics_of_interest = ['mnll', 'jsd']
counts_metrics_of_interest = ['counts-pearson', 'counts-spearman']

"""
Architecture "midpoints" for each different kind of genomics data.
These settings are based on Melanie's best understanding of the kind of 
relevant sequence grammar is captured in fly genomes. She's trained way
too many models on way too many data and these are the parameters that 
optimized models always converge around.
"""
train_settings_dict = {
    'solo_bind' : { #Generic binding model
        'n_dil_layers' : 8,
        'conv1_kernel_size' : 7,
        'profile_kernel_size' : 7,
        'filters' : 64,
        'output_length': 1000,
        'counts_loss_weight' : 100
    }
}

#First-pass models to train
modeling_design_dict = {    

    # # ChIP-nexus data, separated by factor
    # 's2_dfd_hth_ha_nexus': {
    #     'regions': {'peaks': 'bed/all_reproducible_peaks_tiled_expanded.bed'},
    #     'cov': {'dfd': {'pos': 'bw/S2_Dfd_Hth_HA_nexus_combined_positive.bw', 
    #                     'neg': 'bw/S2_Dfd_Hth_HA_nexus_combined_negative.bw'}},
    #     'train_settings_name': 'solo_bind', 'optimize': True
    # },
    # 's2_dfd_hth_exd_nexus': {
    #     'regions': {'peaks': 'bed/all_reproducible_peaks_tiled_expanded.bed'},
    #     'cov': {'dfd': {'pos': 'bw/S2_Dfd_Hth_anti_EXD_nexus_combined_positive.bw', 
    #                     'neg': 'bw/S2_Dfd_Hth_anti_EXD_nexus_combined_negative.bw'}},
    #     'train_settings_name': 'solo_bind', 'optimize': True
    # },

    's2_anthox6a_pbx_flag_nexus': { #flag chip pbx
        'regions': {'peaks': 'bed/all_reproducible_peaks_tiled_expanded.bed'},
        'cov': {'pbx': {'pos': 'bw/S2_AntHOX6a_PBX_Flag_nexus_combined_positive.bw', 
                        'neg': 'bw/S2_AntHOX6a_PBX_Flag_nexus_combined_negative.bw'}},
        'train_settings_name': 'solo_bind', 'optimize': True
    },
    's2_anthox6a_pbx_ha_nexus': { #ha chip pbx
        'regions': {'peaks': 'bed/all_reproducible_peaks_tiled_expanded.bed'},
        'cov': {'pbx': {'pos': 'bw/S2_AntHOX6a_PBX_HA_nexus_combined_positive.bw', 
                        'neg': 'bw/S2_AntHOX6a_PBX_HA_nexus_combined_negative.bw'}},
        'train_settings_name': 'solo_bind', 'optimize': True
    },

    's2_anthox1a_pbx_flag_nexus': { #flag chip pbx
        'regions': {'peaks': 'bed/all_reproducible_peaks_tiled_expanded.bed'},
        'cov': {'pbx': {'pos': 'bw/S2_AntHOX1a_PBX_Flag_nexus_combined_positive.bw', 
                        'neg': 'bw/S2_AntHOX1a_PBX_Flag_nexus_combined_negative.bw'}},
        'train_settings_name': 'solo_bind', 'optimize': True
    },
    's2_anthox1a_pbx_ha_nexus': { #ha chip pbx
        'regions': {'peaks': 'bed/all_reproducible_peaks_tiled_expanded.bed'},
        'cov': {'pbx': {'pos': 'bw/S2_AntHOX1a_PBX_HA_nexus_combined_positive.bw', 
                        'neg': 'bw/S2_AntHOX1a_PBX_HA_nexus_combined_negative.bw'}},
        'train_settings_name': 'solo_bind', 'optimize': True
    }

}

## Modify `modeling_cov_dict` 

Create .json configurations for the different steps for BPReveal training. 

In [4]:
#Prepare .bw configurations. Remember, order matters here.
modeling_cov_dict = {}
for k,v in modeling_design_dict.items():
    bw_dict = {}
    if type(v['cov'])==str:
        bw_dict['prepare_bed_inputs'] = [{'bigwig-names': [v['cov']], 'max-quantile': 1, 'min-counts': 1}]
        bw_dict['prepare_train_inputs'] = [{'bigwig-files': [v['cov']], 'revcomp-task-order': 'auto'}]
    else:
        bw_dict['prepare_bed_inputs'] = [{'bigwig-names': list(v2.values()), 'max-quantile': 1, 'min-counts': 1} for v2 in v['cov'].values()]
        bw_dict['prepare_train_inputs'] = [{'bigwig-files': list(v2.values()), 'revcomp-task-order': 'auto'} for v2 in v['cov'].values()]
    modeling_cov_dict[k] = bw_dict
modeling_cov_dict.keys()

dict_keys(['s2_anthox6a_pbx_flag_nexus', 's2_anthox6a_pbx_ha_nexus', 's2_anthox1a_pbx_flag_nexus', 's2_anthox1a_pbx_ha_nexus'])

# Set up dynamic modeling parameters

For optimization, initialize parameters that will be changing. 

This optimization is meant to answer: how much complexity space needs to be allocated to capture sufficient sequence rules and positional syntax to explain the output signals? Thus, convolutional depth, convolutional filter compelxity, and the Dense layer 1's definition of "motif" scanning all come into play here. As with Brennan and Weilert et al (2023) Dev Cell, we will optimize each parameter independently, relying on the anchor `train_settings_dict` architecture to define the core features. Traditionally, BPNet models are highly stable and do not show massive performance changes unless the parameters are unsuited for the biological complexity. Thus, we seek to (1) find the best performing architecture while simultaneously showcasing (2) BPNet architecture is largely stable across all data types.

In [5]:
#Define architectures that will be optimized for each data type
optimize_architecture_settings_dict = {
    'solo_bind' : { #Generic binding model
        'n_dil_layers' : [7, 8, 9, 10],
        'conv1_kernel_size' : [7],
        'profile_kernel_size' : [7],
        'filters' : [16, 32, 64, 128, 256],
        'counts_loss_weight' : [100]
    }
}

# Train, predict, and assess optimized models

## Prepare training datasets

First, write the .json files required to generate training datasets.

In [6]:
#Loop through different input_lengths based on the different model requirements to generate data.
for model_name,model_bws in modeling_cov_dict.items():

    if modeling_design_dict[model_name]['optimize']:
        #For each model, optimize based on selected parameters
        model_type = modeling_design_dict[model_name]['train_settings_name']
        default_dict = train_settings_dict[model_type]
        opt_dict = optimize_architecture_settings_dict[model_type]
        
        for param_name, param_values in opt_dict.items():
            for i,param_value in enumerate(param_values):
        
                #Overwrite parameter in appropriate field
                current_settings_dict = default_dict.copy()
                current_settings_dict[param_name] = param_value
                max_jitter = int(current_settings_dict['output_length']*.2)
                
                #Assign corresponding input length
                input_length_output = !{python_path} {bpreveal_path}/src/lengthCalc.py \
                    --output-len {current_settings_dict['output_length']} \
                    --n-dil-layers {current_settings_dict['n_dil_layers']} \
                    --conv1-kernel-size {current_settings_dict['conv1_kernel_size']} \
                    --profile-kernel-size {current_settings_dict['profile_kernel_size']}
                input_length = int(input_length_output[0])
                
                #Set up parameter prefix and input length prefix to reference the data and regions to train on.
                param_prefix = model_name + '_' + '_'.join([k + '_' + str(v) for k,v in current_settings_dict.items()])
        
                #Consolidate into a .json-readable dictionary
                prepare_bed_regions_dict = {'heads': model_bws['prepare_bed_inputs'],
                                            'splits': {"test-chroms": test_chroms, 
                                                       "val-chroms": val_chroms,
                                                       "train-chroms": train_chroms,
                                                       "regions": list(modeling_design_dict[model_name]['regions'].values())},
                                            'genome': f'{genome}',
                                            'write-counts-to': f'bed/bpreveal/optimize/{param_prefix}_all.stats',
                                            'output-length': current_settings_dict['output_length'],
                                            'input-length': input_length,
                                            'max-jitter': max_jitter,
                                            'output-prefix': f'bed/bpreveal/optimize/{param_prefix}',
                                            'resize-mode': 'center',
                                            'remove-overlaps': False,
                                            'verbosity': 'INFO'} #Switch this to DEBUG if you are missing regions
                prepare_bed_regions_json = json.dumps(prepare_bed_regions_dict, indent=4)
                bed_json_file = f'json/optimize/prepareBedPeaks_{param_prefix}.json'
                with open(bed_json_file, 'w') as outfile:
                    outfile.write(prepare_bed_regions_json)

                # Prepare input training data
                prepare_input_train_dict = {'genome': genome, 
                                    'input-length': input_length, 
                                    'output-length': current_settings_dict['output_length'], 
                                    'max-jitter': max_jitter, 
                                    'regions': f'bed/bpreveal/optimize/{param_prefix}_train.bed', 
                                    'output-h5': f'input/optimize/{param_prefix}_train.h5', 
                                    'heads': model_bws['prepare_train_inputs'], 
                                    'reverse-complement': reverse_complement,
                                    'verbosity': 'DEBUG'} 
                prepare_input_train_json = json.dumps(prepare_input_train_dict, indent=4)
                train_json_file = f'json/optimize/prepareInputTrain_{param_prefix}.json'        
                with open(train_json_file, 'w') as outfile:
                    outfile.write(prepare_input_train_json)

                # Prepare input validation data
                prepare_input_valid_dict = {'genome': genome, 
                                    'input-length': input_length, 
                                    'output-length': current_settings_dict['output_length'], 
                                    'max-jitter': max_jitter, 
                                    'regions': f'bed/bpreveal/optimize/{param_prefix}_val.bed', 
                                    'output-h5': f'input/optimize/{param_prefix}_val.h5', 
                                    'heads': model_bws['prepare_train_inputs'], 
                                    'reverse-complement': reverse_complement,
                                    'verbosity': 'DEBUG'} 
                prepare_input_valid_json = json.dumps(prepare_input_valid_dict, indent=4)
                valid_json_file = f'json/optimize/prepareInputValid_{param_prefix}.json'        
                with open(valid_json_file, 'w') as outfile:
                    outfile.write(prepare_input_valid_json)

Next, generate SLURM jobs to generate the .h5 and .bed files required for training/validation datasets.

In [7]:
array_cmds = []
for model_name,model_bws in modeling_cov_dict.items():

    if modeling_design_dict[model_name]['optimize']:
        #For each model, optimize based on selected parameters
        model_type = modeling_design_dict[model_name]['train_settings_name']
        default_dict = train_settings_dict[model_type]
        opt_dict = optimize_architecture_settings_dict[model_type]
        
        for param_name, param_values in opt_dict.items():
            for i,param_value in enumerate(param_values):
        
                #Overwrite parameter in appropriate field
                current_settings_dict = default_dict.copy()
                current_settings_dict[param_name] = param_value
                max_jitter = int(current_settings_dict['output_length']*.2)

                #Set up parameter prefix and input length prefix to reference the data and regions to train on.
                param_prefix = model_name + '_' + '_'.join([k + '_' + str(v) for k,v in current_settings_dict.items()])
                
                prepare_peaks_cmd = f'{python_path} {bpreveal_path}/src/prepareBed.py json/optimize/prepareBedPeaks_{param_prefix}.json'
                prepare_train_cmd = f'{python_path} {bpreveal_path}/src/prepareTrainingData.py json/optimize/prepareInputTrain_{param_prefix}.json'
                prepare_valid_cmd = f'{python_path} {bpreveal_path}/src/prepareTrainingData.py json/optimize/prepareInputValid_{param_prefix}.json'

                #Write to a job array
                cmds_str = '\n'.join([prepare_peaks_cmd] + [prepare_train_cmd] + [prepare_valid_cmd])
                array_cmd = [cmds_str]
                array_cmds += array_cmd

generate_slurm_array(cmds = list(set(array_cmds)), 
                     output_file = 'scripts/modeling/optimize_solo_models_prepare_data.slurm',
                     simultaneous_jobs = 20, gpu = False)
print('sbatch scripts/modeling/optimize_solo_models_prepare_data.slurm')

sbatch scripts/modeling/optimize_solo_models_prepare_data.slurm


## Generate training scripts

Here, we will generate training .json configuration files and training scripts for the final training.

In [8]:
array_cmds = []
for model_name,model_info in modeling_design_dict.items():
    tasks = list(model_info['cov'].keys())
    num_channels = len(list(model_info['cov'].values())[0].keys())

    if modeling_design_dict[model_name]['optimize']:
        #For each model, optimize based on selected parameters
        model_type = modeling_design_dict[model_name]['train_settings_name']
        default_dict = train_settings_dict[model_type]
        opt_dict = optimize_architecture_settings_dict[model_type]
        
        for param_name, param_values in opt_dict.items():
            for i,param_value in enumerate(param_values):
        
                #Overwrite parameter in appropriate field
                current_settings_dict = default_dict.copy()
                current_settings_dict[param_name] = param_value
                max_jitter = int(current_settings_dict['output_length']*.2)

                #Assign corresponding input length
                input_length_output = !{python_path} {bpreveal_path}/src/lengthCalc.py \
                    --output-len {current_settings_dict['output_length']} \
                    --n-dil-layers {current_settings_dict['n_dil_layers']} \
                    --conv1-kernel-size {current_settings_dict['conv1_kernel_size']} \
                    --profile-kernel-size {current_settings_dict['profile_kernel_size']}
                input_length = int(input_length_output[0])
                
                #Set up parameter prefix and input length prefix to reference the data and regions to train on.
                param_prefix = model_name + '_' + '_'.join([k + '_' + str(v) for k,v in current_settings_dict.items()])
                
                #Set up training parameters for the base model
                train_dict = {'settings': {'output-prefix': f'models/optimize/{param_prefix}', 
                                           'epochs': 200, 
                                           'early-stopping-patience': early_stopping_patience, 
                                           'batch-size': batch_size, 
                                           'learning-rate': learning_rate, 
                                           'learning-rate-plateau-patience': 5, 
                                           'max-jitter': max_jitter, 
                                           'architecture': {'architecture-name': 'bpnet', 
                                                            'input-length': input_length, 
                                                            'output-length': current_settings_dict['output_length'], 
                                                            'model-name': param_prefix, 
                                                            'model-args': '', 
                                                            'filters': current_settings_dict['filters'], 
                                                            'layers': current_settings_dict['n_dil_layers'], 
                                                            'input-filter-width': current_settings_dict['conv1_kernel_size'], 
                                                            'output-filter-width': current_settings_dict['profile_kernel_size']}}, 
                              'heads': [{'num-tasks': num_channels, 
                                     'profile-loss-weight': 1, 
                                     'head-name': task, 
                                     'counts-loss-frac-target': .1,
                                     'counts-loss-weight': current_settings_dict['counts_loss_weight']} for task in tasks],
                              'train-data': f'input/optimize/{param_prefix}_train.h5', 
                              'val-data': f'input/optimize/{param_prefix}_val.h5', 
                              'verbosity': 'WARNING'}

                train_json = json.dumps(train_dict, indent=4)
                train_json_file = f'json/optimize/trainSoloModel_{param_prefix}.json'
                with open(train_json_file, 'w') as outfile:
                    outfile.write(train_json)
                    
                #Set up prediction parameters
                pred_dict = {'settings': {'genome': genome, 
                                          'output-h5': f'preds/optimize/{param_prefix}_test.h5', 
                                          'batch-size': batch_size, 
                                          'heads': len(tasks), 
                                          'architecture': {'model-file': f'models/optimize/{param_prefix}.keras', 
                                                           'input-length': input_length, 
                                                           'output-length': current_settings_dict['output_length']}}, 
                             'bed-file': f'bed/bpreveal/optimize/{param_prefix}_test.bed',
                             'verbosity': 'DEBUG'}
                pred_json = json.dumps(pred_dict, indent=4)
                pred_json_file = f'json/optimize/predictPeaks_{param_prefix}.json'
                with open(pred_json_file, 'w') as outfile:
                    outfile.write(pred_json)
                
                #Set up training and prediction commands
                model_train = [f'{python_path} {bpreveal_path}/src/trainSoloModel.py json/optimize/trainSoloModel_{param_prefix}.json']
                model_predict = [f'{python_path} {bpreveal_path}/src/makePredictionsBed.py json/optimize/predictPeaks_{param_prefix}.json']
                    
                #Write to a job array
                cmds_str = '\n'.join(model_train + model_predict)
                array_cmd = [cmds_str]
                array_cmds += array_cmd

generate_slurm_array(cmds = list(set(array_cmds)), 
                     output_file = 'scripts/modeling/optimize_solo_models_train.slurm', mem=80)
print('sbatch scripts/modeling/optimize_solo_models_train.slurm')

sbatch scripts/modeling/optimize_solo_models_train.slurm


## Generate prediction .bw files and assess performance

Here, we will extract .bw coverage from the generated .h5 files and compare them to generated features in order to compute performance metrics

In [9]:
array_cmds = []
for model_name,model_info in modeling_design_dict.items():

    if modeling_design_dict[model_name]['optimize']:
        #For each model, optimize based on selected parameters
        model_type = modeling_design_dict[model_name]['train_settings_name']
        default_dict = train_settings_dict[model_type]
        opt_dict = optimize_architecture_settings_dict[model_type]
        
        for param_name, param_values in opt_dict.items():
            for i,param_value in enumerate(param_values):
        
                #Overwrite parameter in appropriate field
                current_settings_dict = default_dict.copy()
                current_settings_dict[param_name] = param_value
                
                #Set up parameter prefix and input length prefix to reference the data and regions to train on.
                param_prefix = model_name + '_' + '_'.join([k + '_' + str(v) for k,v in current_settings_dict.items()])

                for head_counter, (task, cov) in enumerate(model_info['cov'].items()):
                    for channel_counter, (channel_name, channel) in enumerate(cov.items()):

                        predict_cmd = f'{python_path} {bpreveal_path}/src/predictToBigwig.py \
                        --h5 preds/optimize/{param_prefix}_test.h5 \
                        --bw preds/optimize/{param_prefix}_{task}_{channel_name}_test.bw \
                        --head-id {head_counter} \
                        --task-id {channel_counter} --mode profile --verbose'

                        #Write to a job array
                        cmds_str = '\n'.join([predict_cmd])
                        array_cmd = [cmds_str]
                        array_cmds += array_cmd

generate_slurm_array(cmds = list(set(array_cmds)), 
                     output_file = 'scripts/modeling/optimize_solo_models_predict.slurm',
                     simultaneous_jobs = 20, gpu = False)
print('sbatch scripts/modeling/optimize_solo_models_predict.slurm')

sbatch scripts/modeling/optimize_solo_models_predict.slurm


## Compute performance metrics

Here, we will compute and compare performance metrics on the test set of chromosomes. We will assess the 50th percentile as the key feature of profile metrics as well as the overall total counts performance. 

First, calculate performance metrics and save in a .json file.

In [10]:
array_cmds = []
for model_name,model_info in modeling_design_dict.items():
    if modeling_design_dict[model_name]['optimize']:
        #For each model, optimize based on selected parameters
        model_type = modeling_design_dict[model_name]['train_settings_name']
        default_dict = train_settings_dict[model_type]
        opt_dict = optimize_architecture_settings_dict[model_type]
        
        for param_name, param_values in opt_dict.items():
            for i,param_value in enumerate(param_values):
        
                #Overwrite parameter in appropriate field
                current_settings_dict = default_dict.copy()
                current_settings_dict[param_name] = param_value
                
                #Set up parameter prefix and input length prefix to reference the data and regions to train on.
                param_prefix = model_name + '_' + '_'.join([k + '_' + str(v) for k,v in current_settings_dict.items()])

                #Assess test regions
                peaks_path = f'bed/bpreveal/optimize/{param_prefix}_test.bed'
                for head_counter, (task, cov) in enumerate(model_info['cov'].items()):
                    for channel_counter, (channel_name, channel) in enumerate(cov.items()):
    
                        reference_path = channel
                        preds_path = f'preds/optimize/{param_prefix}_{task}_{channel_name}_test.bw'
                        
                        #Compute correlations and profile metrics over quantiles
                        metrics_json_file = f'json/optimize/metrics_{param_prefix}_{task}_{channel_name}_test.json'

                        metrics_cmd = f'{python_path} {bpreveal_path}/src/metrics.py \
                            --reference {reference_path} \
                            --pred {preds_path} \
                            --regions {peaks_path} \
                            --threads 10 --json-output --apply-abs --skip-zeroes | tee {metrics_json_file}'
                        
                        #Write to a job array
                        cmds_str = '\n'.join([metrics_cmd])
                        array_cmd = [cmds_str]
                        array_cmds += array_cmd
                        
generate_slurm_array(cmds = list(set(array_cmds)), 
                     output_file = 'scripts/modeling/optimize_solo_models_metrics.slurm',
                     simultaneous_jobs = 20, gpu = False)
print('sbatch scripts/modeling/optimize_solo_models_metrics.slurm')

sbatch scripts/modeling/optimize_solo_models_metrics.slurm


Next, consolidate performance metrics by the different parameters and save into a tidy .tsv file. 

In [11]:
metrics_df = pd.DataFrame()
for model_name,model_info in modeling_design_dict.items():
    if modeling_design_dict[model_name]['optimize']:
        #For each model, optimize based on selected parameters
        model_type = modeling_design_dict[model_name]['train_settings_name']
        default_dict = train_settings_dict[model_type]
        opt_dict = optimize_architecture_settings_dict[model_type]
        
        for param_name, param_values in opt_dict.items():
            for i,param_value in enumerate(param_values):
        
                #Overwrite parameter in appropriate field
                current_settings_dict = default_dict.copy()
                current_settings_dict[param_name] = param_value
                
                #Set up parameter prefix and input length prefix to reference the data and regions to train on.
                param_prefix = model_name + '_' + '_'.join([k + '_' + str(v) for k,v in current_settings_dict.items()])

                peaks_path = f'bed/bpreveal/optimize/{param_prefix}_test.bed'
                for head_counter, (task, cov) in enumerate(model_info['cov'].items()):
                    for channel_counter, (channel_name, channel) in enumerate(cov.items()):
    
                        reference_path = channel
                        preds_path = f'preds/optimize/{param_prefix}_{task}_{channel_name}_test.bw'
                        
                        #Compute correlations and profile metrics over quantiles
                        metrics_json_file = f'json/optimize/metrics_{param_prefix}_{task}_{channel_name}_test.json'
                        with open(metrics_json_file) as f:
                            metrics_dict = json.load(f)
    
                        df = pd.DataFrame([os.path.basename(metrics_dict['predicted'])]).transpose()
                        df.columns = ['bw']
                        
                        df[['model', 'bw']] = df['bw'].str.split('_n_dil_layers_', n=1, expand=True)
                        df[['n_dil_layers', 'bw']] = df['bw'].str.split('_conv1_kernel_size_', n=1, expand=True)
                        df[['conv_kernel_size', 'bw']] = df['bw'].str.split('_profile_kernel_size_', n=1, expand=True)
                        df[['profile_kernel_size', 'bw']] = df['bw'].str.split('_filters_', n=1, expand=True)
                        df[['filters', 'bw']] = df['bw'].str.split('_output_length_1000_counts_loss_weight_', n=1, expand=True)
                        df[['counts_loss_weight', 'bw']] = df['bw'].str.split('_', n=1, expand=True)
                        df[['task', 'bw']] = df['bw'].str.split('_', n=1, expand=True)
                        df[['channel', 'bw']] = df['bw'].str.split('_', n=1, expand=True)
                        df[['filler', 'bw']] = df['bw'].str.split(f'.bw', n=1, expand=True)
                        df['dataset'] = 'test'
                        df = df.drop('bw', axis = 1)
                        df[profile_metrics_of_interest] = [np.median(metrics_dict[metric]['quantiles']) for metric in profile_metrics_of_interest]
                        df[counts_metrics_of_interest] = [metrics_dict[metric] for metric in counts_metrics_of_interest]
                        metrics_df = pd.concat([metrics_df, df])

metrics_df.to_csv('tsv/performance_metrics_across_solo_optimized.tsv.gz', sep = '\t', index = False)

In [12]:
pd.set_option('display.max_rows', None)
metrics_df[metrics_df.channel=='pos']

,model,n_dil_layers,conv_kernel_size,profile_kernel_size,filters,counts_loss_weight,task,channel,filler,dataset,mnll,jsd,counts-pearson,counts-spearman
0,s2_anthox6a_pbx_flag_nexus,7,7,7,64,100,pbx,pos,test,test,-645.740057,0.593586,0.090396,0.522486
0,s2_anthox6a_pbx_flag_nexus,8,7,7,64,100,pbx,pos,test,test,-647.152373,0.593204,0.067053,0.494662
0,s2_anthox6a_pbx_flag_nexus,9,7,7,64,100,pbx,pos,test,test,-646.237463,0.594566,0.073614,0.532826
0,s2_anthox6a_pbx_flag_nexus,10,7,7,64,100,pbx,pos,test,test,-645.860384,0.592868,0.080372,0.609646
0,s2_anthox6a_pbx_flag_nexus,8,7,7,64,100,pbx,pos,test,test,-647.152373,0.593204,0.067053,0.494662
0,s2_anthox6a_pbx_flag_nexus,8,7,7,64,100,pbx,pos,test,test,-647.152373,0.593204,0.067053,0.494662
0,s2_anthox6a_pbx_flag_nexus,8,7,7,16,100,pbx,pos,test,test,-649.733445,0.596843,0.058866,0.433263
0,s2_anthox6a_pbx_flag_nexus,8,7,7,32,100,pbx,pos,test,test,-646.548262,0.593862,0.061327,0.525880
0,s2_anthox6a_pbx_flag_nexus,8,7,7,64,100,pbx,pos,test,test,-647.152373,0.593204,0.067053,0.494662
0,s2_anthox6a_pbx_flag_nexus,8,7,7,128,100,pbx,pos,test,test,-647.385676,0.593114,0.069319,0.524045


## Optimization conclusions

From the resulting performance metrics, we conclude:

+ Binding model:
    + Similar to our old binding models, our current binding model performs well with 9 dilational layers (`n_dil_layers`=9 without `output_length`=1000). Model performance is pretty stable across all values. 
    + Similar to our old binding models, our current binding model performs well with `filters` from around 64. 

# Train, predict, assess and SHAP final models

After optimization, we have architectures that are well-suited for our different types of models and training sets. Thus, we can train each model set over three different folds of chromosomes to ensure that models are stable and consistently returning the same sequence grammar. 

## Re-define optimized training architectures

Since within different groups of data types (MNase, binding, accessibility etc), we are only training one or two cell states of very similar sequence complexity, we will redefine the optimized architectures here.

In [13]:
train_settings_dict = {
    'solo_bind' : { #Generic binding model
        'n_dil_layers' : 9,
        'conv1_kernel_size' : 7,
        'profile_kernel_size' : 7,
        'filters' : 64,
        'output_length': 1000,
        'counts_loss_weight' : 100
    }
}

#Calculate associate input lengths
input_length_dict = {}
for k,v in train_settings_dict.items():
    proxy_length = !{python_path} {bpreveal_path}/src/lengthCalc.py \
        --output-len {v['output_length']} \
        --n-dil-layers {v['n_dil_layers']} \
        --conv1-kernel-size {v['conv1_kernel_size']} \
        --profile-kernel-size {v['profile_kernel_size']}
    input_length_dict[k] = int(proxy_length[0])
input_length_dict

{'solo_bind': 3056}

## Define random sequences

In [14]:
#Calculate associate input lengths
for k,v in train_settings_dict.items():
    proxy_length = !{python_path} {bpreveal_path}/src/lengthCalc.py \
        --output-len {v['output_length']} \
        --n-dil-layers {v['n_dil_layers']} \
        --conv1-kernel-size {v['conv1_kernel_size']} \
        --profile-kernel-size {v['profile_kernel_size']}
    input_length_dict[k] = int(proxy_length[0])
input_length_dict

{'solo_bind': 3056}

In [15]:
for k,v in input_length_dict.items():
    random_seqs_path = f'npz/random_seqs_seed_{seed}_input_{v}_trials_{trials}_array.npz'
    seqs = [np.random.choice(['A', 'C','G', 'T'], v) for i in range(trials)]
    seqs_1he = one_hot_encode_sequences(seqs)
    np.savez(random_seqs_path, seqs_1he = seqs_1he)

## Prepare 3-fold region datasets

Define different chromosome folds for training.

In [16]:
chromosome_folds_dict = {
    'fold1': {'train' : train_chroms,
              'val' : val_chroms,
              'test' : test_chroms},
    'fold2': {'train' : ['chrX', 'chr3R', 'chr2R'],
              'val' : ['chr2L'],
              'test' : ['chr3L']},
    'fold3': {'train' : ['chrX', 'chr3R', 'chr2L'],
              'val' : ['chr3L'],
              'test' : ['chr2R']}
}

## Prepare training datasets

First, write the .json files required to generate training datasets.

In [17]:
#Loop through different input_lengths based on the different model requirements to generate data.
for model_name,model_bws in modeling_cov_dict.items():
    for fold_name,fold_chroms in chromosome_folds_dict.items():        
        fold_prefix = model_name + '_' + str(fold_name)
        input_length = input_length_dict[modeling_design_dict[model_name]['train_settings_name']]
        output_length = train_settings_dict[modeling_design_dict[model_name]['train_settings_name']]['output_length']
        max_jitter = int(output_length*.2)
    
        #Consolidate into a .json-readable dictionary
        prepare_bed_regions_dict = {'heads': model_bws['prepare_bed_inputs'],
                                    'splits': {"test-chroms": fold_chroms['test'], 
                                               "val-chroms": fold_chroms['val'],
                                               "train-chroms": fold_chroms['train'],
                                               "regions": list(modeling_design_dict[model_name]['regions'].values())},
                                    'genome': f'{genome}',
                                    'write-counts-to': f'bed/bpreveal/{fold_prefix}_all.stats',
                                    'output-length': output_length,
                                    'input-length': input_length,
                                    'max-jitter': max_jitter,
                                    'output-prefix': f'bed/bpreveal/{fold_prefix}',
                                    'resize-mode': 'center',
                                    'remove-overlaps': False,
                                    'verbosity': 'INFO'} #Switch this to DEBUG if you are missing regions
        prepare_bed_regions_json = json.dumps(prepare_bed_regions_dict, indent=4)
        bed_json_file = f'json/prepareBedPeaks_{fold_prefix}.json'
        with open(bed_json_file, 'w') as outfile:
            outfile.write(prepare_bed_regions_json)


        # Prepare input training data
        prepare_input_train_dict = {'genome': genome, 
                            'input-length': input_length, 
                            'output-length': output_length, 
                            'max-jitter': max_jitter, 
                            'regions': f'bed/bpreveal/{fold_prefix}_train.bed', 
                            'output-h5': f'input/{fold_prefix}_train.h5', 
                            'heads': model_bws['prepare_train_inputs'], 
                            'reverse-complement': reverse_complement,
                            'verbosity': 'DEBUG'} 
        prepare_input_train_json = json.dumps(prepare_input_train_dict, indent=4)
        train_json_file = f'json/prepareInputTrain_{fold_prefix}.json'        
        with open(train_json_file, 'w') as outfile:
            outfile.write(prepare_input_train_json)
            
        # Prepare validation training data
        prepare_input_valid_dict = {'genome': genome, 
                    'input-length': input_length, 
                    'output-length': output_length, 
                    'max-jitter': max_jitter, 
                    'regions': f'bed/bpreveal/{fold_prefix}_val.bed', 
                    'output-h5': f'input/{fold_prefix}_val.h5', 
                    'heads': model_bws['prepare_train_inputs'], 
                    'reverse-complement': reverse_complement,
                    'verbosity': 'DEBUG'} 
        prepare_input_valid_json = json.dumps(prepare_input_valid_dict, indent=4)
        valid_json_file = f'json/prepareInputValid_{fold_prefix}.json'        
        with open(valid_json_file, 'w') as outfile:
            outfile.write(prepare_input_valid_json)

Next, generate SLURM jobs to generate the .h5 and .bed files required for training/validation datasets.

In [18]:
array_cmds = []
for model_name,model_bws in modeling_cov_dict.items():
    for fold_name,fold_chroms in chromosome_folds_dict.items():        
        fold_prefix = model_name + '_' + str(fold_name)
        
        prepare_peaks_cmd = f'{python_path} {bpreveal_path}/src/prepareBed.py json/prepareBedPeaks_{fold_prefix}.json'
        prepare_train_cmd = f'{python_path} {bpreveal_path}/src/prepareTrainingData.py json/prepareInputTrain_{fold_prefix}.json'
        prepare_valid_cmd = f'{python_path} {bpreveal_path}/src/prepareTrainingData.py json/prepareInputValid_{fold_prefix}.json'

        #Write to a job array
        cmds_str = '\n'.join([prepare_peaks_cmd] + [prepare_train_cmd] + [prepare_valid_cmd])
        array_cmd = [cmds_str]
        array_cmds += array_cmd

generate_slurm_array(cmds = array_cmds, output_file = 'scripts/modeling/bpnet_solo_models_prepare_data.slurm',
                     simultaneous_jobs = 20, mem = 20, gpu = False)
print('sbatch scripts/modeling/bpnet_solo_models_prepare_data.slurm')

sbatch scripts/modeling/bpnet_solo_models_prepare_data.slurm


## Generate training scripts

Here, we will generate training .json configuration files and training scripts for the final training.

In [19]:
array_cmds = []
for model_name,model_info in modeling_design_dict.items():
    tasks = list(model_info['cov'].keys())
    num_channels = len(list(model_info['cov'].values())[0].keys())
    train_settings = train_settings_dict[model_info['train_settings_name']]
    input_length = input_length_dict[model_info['train_settings_name']]
                       
    for fold_name,fold_chroms in chromosome_folds_dict.items():
        
        fold_prefix = model_name + '_' + str(fold_name)

        #Set up training parameters for the base model
        train_dict = {'settings': {'output-prefix': f'models/{fold_prefix}', 
                                   'epochs': 200, 
                                   'early-stopping-patience': early_stopping_patience, 
                                   'batch-size': batch_size, 
                                   'learning-rate': learning_rate, 
                                   'learning-rate-plateau-patience': 5, 
                                   'max-jitter': max_jitter, 
                                   'architecture': {'architecture-name': 'bpnet', 
                                                    'input-length': input_length, 
                                                    'output-length': output_length, 
                                                    'model-name': fold_prefix, 
                                                    'model-args': '', 
                                                    'filters': train_settings['filters'], 
                                                    'layers': train_settings['n_dil_layers'], 
                                                    'input-filter-width': train_settings['conv1_kernel_size'], 
                                                    'output-filter-width': train_settings['profile_kernel_size']}}, 
                      'heads': [{'num-tasks': num_channels, 
                             'profile-loss-weight': 1, 
                             'head-name': task, 
                             'counts-loss-frac-target': .1,
                             'counts-loss-weight': train_settings['counts_loss_weight']} for task in tasks],
                      'train-data': f'input/{fold_prefix}_train.h5', 
                      'val-data': f'input/{fold_prefix}_val.h5', 
                      'verbosity': 'WARNING'}

        train_json = json.dumps(train_dict, indent=4)
        train_json_file = f'json/trainSoloModel_{fold_prefix}.json'
        with open(train_json_file, 'w') as outfile:
            outfile.write(train_json)
            
        #Set up prediction parameters
        pred_dict = {'settings': {'genome': genome, 
                                  'output-h5': f'preds/{fold_prefix}_all.h5', 
                                  'batch-size': batch_size, 
                                  'heads': len(tasks), 
                                  'architecture': {'model-file': f'models/{fold_prefix}.keras', 
                                                   'input-length': input_length, 
                                                   'output-length': output_length}}, 
                     'bed-file': f'bed/bpreveal/{fold_prefix}_all.bed',
                     'verbosity': 'INFO'}
        pred_json = json.dumps(pred_dict, indent=4)
        pred_json_file = f'json/predictPeaks_{fold_prefix}.json'
        with open(pred_json_file, 'w') as outfile:
            outfile.write(pred_json)
        
        #Set up training and prediction commands
        model_train = [f'{python_path} {bpreveal_path}/src/trainSoloModel.py json/trainSoloModel_{fold_prefix}.json']
        model_predict = [f'{python_path} {bpreveal_path}/src/makePredictionsBed.py json/predictPeaks_{fold_prefix}.json']
            
        #Write to a job array
        cmds_str = '\n'.join(model_train + model_predict)
        array_cmd = [cmds_str]
        array_cmds += array_cmd

generate_slurm_array(cmds = array_cmds, output_file = 'scripts/modeling/bpnet_solo_models_train.slurm')
print('sbatch scripts/modeling/bpnet_solo_models_train.slurm')

sbatch scripts/modeling/bpnet_solo_models_train.slurm


## Generate prediction .bw files and assess performance

Here, we will extract .bw coverage from the generated .h5 files and compare them to generated features in order to compute performance metrics

In [20]:
array_cmds = []
for model_name,model_info in modeling_design_dict.items():               
    for fold_name,fold_chroms in chromosome_folds_dict.items():
        fold_prefix = model_name + '_' + str(fold_name)
        for head_counter, (task, cov) in enumerate(model_info['cov'].items()):
            for channel_counter, (channel_name, channel) in enumerate(cov.items()):

                predict_cmd = f'{python_path} {bpreveal_path}/src/predictToBigwig.py \
                --h5 preds/{fold_prefix}_all.h5 \
                --bw preds/{fold_prefix}_{task}_{channel_name}_all.bw \
                --head-id {head_counter} \
                --task-id {channel_counter} --mode profile --verbose'

                #Write to a job array
                cmds_str = '\n'.join([predict_cmd])
                array_cmd = [cmds_str]
                array_cmds += array_cmd

generate_slurm_array(cmds = array_cmds, output_file = 'scripts/modeling/bpnet_solo_models_predict.slurm',
                     simultaneous_jobs = 20, gpu = False)
print('sbatch scripts/modeling/bpnet_solo_models_predict.slurm')

sbatch scripts/modeling/bpnet_solo_models_predict.slurm


## Compute performance metrics

Here, we will compute and compare performance metrics on the test set of chromosomes. We will assess the 50th percentile as the key feature of profile metrics as well as the overall total counts performance. 

First, calculate performance metrics and save in a .json file.

In [21]:
array_cmds = []
for model_name,model_info in modeling_design_dict.items():
    for fold_name,fold_chroms in chromosome_folds_dict.items():
        fold_prefix = model_name + '_' + str(fold_name)
        for dataset in ['test','val','train']:
            peaks_path = f'bed/bpreveal/{fold_prefix}_{dataset}.bed'
            
            for head_counter, (task, cov) in enumerate(model_info['cov'].items()):
                for channel_counter, (channel_name, channel) in enumerate(cov.items()):

                    reference_path = channel
                    preds_path = f'preds/{fold_prefix}_{task}_{channel_name}_all.bw'
                    
                    #Compute correlations and profile metrics over quantiles
                    metrics_json_file = f'json/metrics_{fold_prefix}_{task}_{channel_name}_peaks_{dataset}.json'
                    metrics_cmd = f'{python_path} {bpreveal_path}/src/metrics.py \
                        --reference {reference_path} \
                        --pred {preds_path} \
                        --regions {peaks_path} \
                        --threads 70 --json-output --apply-abs --skip-zeroes | tee {metrics_json_file}'
                    
                    #Write to a job array
                    cmds_str = '\n'.join([metrics_cmd])
                    array_cmd = [cmds_str]
                    array_cmds += array_cmd
                        
generate_slurm_array(cmds = array_cmds, output_file = 'scripts/modeling/bpnet_solo_models_metrics.slurm',
                     simultaneous_jobs = 20, mem = 10, gpu = False)
print('sbatch scripts/modeling/bpnet_solo_models_metrics.slurm')

sbatch scripts/modeling/bpnet_solo_models_metrics.slurm


## Format performance metrics

Next, consolidate performance metrics by the different parameters and save into a tidy .tsv file. 

In [22]:
metrics_df = pd.DataFrame()
for model_name,model_info in modeling_design_dict.items():
    for fold_name,fold_chroms in chromosome_folds_dict.items():
        fold_prefix = model_name + '_' + str(fold_name)
        for dataset in ['test','val','train']:
            peaks_path = f'bed/bpreveal/{fold_prefix}_{dataset}.bed'
            for head_counter, (task, cov) in enumerate(model_info['cov'].items()):
                for channel_counter, (channel_name, channel) in enumerate(cov.items()):

                    reference_path = channel
                    preds_path = f'preds/{fold_prefix}_{task}_{channel_name}_all.bw'
                    
                    #Read in performance metrics
                    metrics_json_file = f'json/metrics_{fold_prefix}_{task}_{channel_name}_peaks_{dataset}.json'
                    with open(metrics_json_file) as f:
                        metrics_dict = json.load(f)

                    #Format into a tidy df
                    df = pd.DataFrame([os.path.basename(metrics_dict['predicted'])]).transpose()
                    df.columns = ['bw']
                    df[['model', 'bw']] = df['bw'].str.split('_fold', n=1, expand=True)
                    df[['fold', 'bw']] = df['bw'].str.split('_', n=1, expand=True)
                    df[['task', 'bw']] = df['bw'].str.split('_', n=1, expand=True)
                    df[['channel', 'bw']] = df['bw'].str.split('_', n=1, expand=True)
                    df[['filler', 'bw']] = df['bw'].str.split(f'.bw', n=1, expand=True)
                    df['dataset'] = dataset
                    df = df.drop('bw', axis = 1)
                    df[profile_metrics_of_interest] = [np.median(metrics_dict[metric]['quantiles']) for metric in profile_metrics_of_interest]
                    df[counts_metrics_of_interest] = [metrics_dict[metric] for metric in counts_metrics_of_interest]
                    metrics_df = pd.concat([metrics_df, df])

metrics_df.to_csv('tsv/performance_metrics_across_solo_train.tsv.gz', sep = '\t', index = False)

In [23]:
metrics_df = pd.read_csv('tsv/performance_metrics_across_solo_train.tsv.gz', sep = '\t')
metrics_df[(metrics_df.dataset=='test') & (metrics_df.channel=='pos')]

,model,fold,task,channel,filler,dataset,mnll,jsd,counts-pearson,counts-spearman
0,s2_anthox6a_pbx_flag_nexus,1,pbx,pos,all,test,-645.895995,0.591528,0.084168,0.570389
6,s2_anthox6a_pbx_flag_nexus,2,pbx,pos,all,test,-869.773946,0.529807,0.393062,0.529335
12,s2_anthox6a_pbx_flag_nexus,3,pbx,pos,all,test,-988.327759,0.498630,0.542661,0.509502
18,s2_anthox6a_pbx_ha_nexus,1,pbx,pos,all,test,-757.631310,0.540600,0.067175,0.358737
24,s2_anthox6a_pbx_ha_nexus,2,pbx,pos,all,test,-996.992133,0.483333,0.476405,0.561988
30,s2_anthox6a_pbx_ha_nexus,3,pbx,pos,all,test,-1150.701600,0.450459,0.457813,0.534586
36,s2_anthox1a_pbx_flag_nexus,1,pbx,pos,all,test,-426.665578,0.680820,0.030348,0.327167
42,s2_anthox1a_pbx_flag_nexus,2,pbx,pos,all,test,-595.115393,0.633849,0.327282,0.395937
48,s2_anthox1a_pbx_flag_nexus,3,pbx,pos,all,test,-673.088073,0.609126,0.640518,0.444003
54,s2_anthox1a_pbx_ha_nexus,1,pbx,pos,all,test,-519.413732,0.635801,0.095468,0.551493


These performance metrics will be more directly compared in the next .Rmd. 

## Calculate SHAP scores for all models

### Generate SHAP scores

In [24]:
array_cmds = []

for model_name,model_info in modeling_design_dict.items():
    num_channels = len(list(model_info['cov'].values())[0].keys())
    input_length = input_length_dict[model_info['train_settings_name']]
    
    for fold_name,fold_chroms in chromosome_folds_dict.items():
        fold_prefix = model_name + '_' + str(fold_name)

        #Set up SHAP parameters for each task.
        shap_template_dict = {
            'genome': genome,
            'bed-file': f'bed/bpreveal/{fold_prefix}_all.bed',
            'input-length': input_length,
            'output-length': output_length,
            'num-shuffles': 20,
            'verbosity': 'DEBUG'
        }
        
        shap_dict = shap_template_dict.copy()
        shap_dict['model-file'] = f'models/{fold_prefix}.keras'
        for head_counter, (task, cov) in enumerate(model_info['cov'].items()):
            shap_dict['heads'] = len(list(model_info['cov'].keys()))
            shap_dict['head-id'] = head_counter
            shap_dict['profile-task-ids'] = list(range(num_channels))
            shap_dict['profile-h5'] = f'shap/{fold_prefix}_{task}_profile.h5'
            shap_dict['counts-h5'] = f'shap/{fold_prefix}_{task}_counts.h5'
            shap_json = json.dumps(shap_dict, indent=4)
            shap_json_file = f'json/shapFlat_{fold_prefix}_{task}.json'
            with open(shap_json_file, 'w') as outfile:
                outfile.write(shap_json)   

            #Set up the SHAP commands
            model_shap = [f'{python_path} {bpreveal_path}/src/interpretFlat.py json/shapFlat_{fold_prefix}_{task}.json']
                
            #Write to a job array
            cmds_str = '\n'.join(model_shap)
            array_cmd = [cmds_str]
            array_cmds += array_cmd

generate_slurm_array(cmds = array_cmds, output_file = 'scripts/modeling/bpnet_solo_models_shap.slurm')
print('sbatch scripts/modeling/bpnet_solo_models_shap.slurm')

sbatch scripts/modeling/bpnet_solo_models_shap.slurm


### Convert SHAP .h5 to .bw

In [25]:
array_cmds = []
for model_name,model_info in modeling_design_dict.items():
    for fold_name,fold_chroms in chromosome_folds_dict.items():
        fold_prefix = model_name + '_' + str(fold_name)
        for task in list(model_info['cov'].keys()):

            counts_shap_cmd = f'{python_path} {bpreveal_path}/src/shapToBigwig.py \
            --h5 shap/{fold_prefix}_{task}_counts.h5 \
            --bw shap/{fold_prefix}_{task}_counts.bw \
            --verbose'

            profile_shap_cmd = f'{python_path} {bpreveal_path}/src/shapToBigwig.py \
            --h5 shap/{fold_prefix}_{task}_profile.h5 \
            --bw shap/{fold_prefix}_{task}_profile.bw \
            --verbose'

            #Write to a job array
            cmds_str = '\n'.join([profile_shap_cmd] + [counts_shap_cmd])
            array_cmd = [cmds_str]
            array_cmds += array_cmd

generate_slurm_array(cmds = array_cmds, output_file = 'scripts/modeling/bpnet_solo_models_shap_to_bw.slurm',
                     simultaneous_jobs = 20, gpu = False)
print('sbatch scripts/modeling/bpnet_solo_models_shap_to_bw.slurm')

sbatch scripts/modeling/bpnet_solo_models_shap_to_bw.slurm


These performance metrics will be more directly compared in the next .Rmd. But we can already see that retraining did not add any additional information to these data.

# Validate all final models with marginalization

Marginalize models with designated sequences to assess whether the models are showing expected coverages across. For example, we would expect ATAC-seq bias models to not respond to given binding motifs, nor would we expect injection of pioneering motif sequences to deplete a nucleosome across MNase intrinsic models. But we would expect binding to a TF binding motif for a solo model. 

This can be a separate assessment that lies adjacent to running TF-MoDISco later to quickly ensure that our models are learning the correct sequence grammar. While simple, it is surprisingly direct and effective.

## Select motifs to marginalize

Here, we will select motifs to marginalize across.

In [26]:
motifs_dict = {
    'AbdB': 'TTTATGA', 
    'Anthox6a': 'ATGATTGATGG', 
    'Anthox1a': 'ATGATTTATGG', 
    'Dfd': 'TTAATGGC'
    
}

## Predict marginalizations

In [27]:
null_pred_df = pd.DataFrame()
injected_pred_df = pd.DataFrame()

for model_name,model_info in modeling_design_dict.items():
    
    tasks = list(model_info['cov'].keys())
    num_channels = len(list(model_info['cov'].values())[0].keys())
    input_length = input_length_dict[model_info['train_settings_name']]
    random_seqs = [one_hot_decode_sequence(i) for i in np.load(f'npz/random_seqs_seed_{seed}_input_{input_length}_trials_{trials}_array.npz')['seqs_1he']]

    for fold_name,fold_chroms in chromosome_folds_dict.items():
        fold_prefix = model_name + '_' + str(fold_name)
    
        #Import model
        model_dir = f'models/{fold_prefix}.keras'
        model = load_model(model_dir, custom_objects = {'multinomialNll' : losses.multinomialNll, "reweightableMse": losses.dummyMse})
        tasks_n = len(tasks)
        
        #Predict null accessibility
        null_preds_arr = model.predict(one_hot_encode_sequences(random_seqs), verbose = False)
        
        #Convert logits and logcounts to human-readable ChIP-nexus profile with counts
        for i,task in enumerate(tasks):
            profile_by_task = []
            for j in range(trials):
                profile = logitsToProfile(logitsAcrossSingleRegion = null_preds_arr[i][j],
                                          logCountsAcrossSingleRegion = null_preds_arr[i+tasks_n][j])
                profile_by_task.append(profile)
            
            #Average across trials
            null_pred_avg_arr = np.mean(np.array(profile_by_task), axis = 0)
            
            #Convert to tidy pd.df
            null_pred_avg_df = pd.DataFrame(null_pred_avg_arr, columns = list(model_info['cov'][task].keys()))
            null_pred_avg_df['position'] = list(range(null_pred_avg_df.shape[0]))
            null_pred_avg_df = null_pred_avg_df.melt(id_vars = 'position', var_name = 'channel', value_name = 'pred')
            null_pred_avg_df['model_name'] = fold_prefix    
            null_pred_avg_df['task'] = task
            null_pred_df = pd.concat([null_pred_df, null_pred_avg_df])
    
        for motif,motif_seq in motifs_dict.items():
        
            #Inject motif into null sequences
            injected_seqs = []
            for k in range(trials):
                injected_seq = insert_motif(seq = random_seqs[k], motif = motif_seq, position = input_length // 2)
                injected_seqs.append(injected_seq)
            injected_seqs_1he = one_hot_encode_sequences(injected_seqs)
            
            #Predict injected sequences
            injected_preds_raw_arr = model.predict(injected_seqs_1he, verbose = False)
        
            #Convert logits and logcounts to human-readable ChIP-nexus profile with counts
            for i,task in enumerate(tasks):
                profile_by_task = []
                for j in range(trials):
                    profile = logitsToProfile(logitsAcrossSingleRegion = injected_preds_raw_arr[i][j],
                                              logCountsAcrossSingleRegion = injected_preds_raw_arr[i+tasks_n][j])
                    profile_by_task.append(profile)
            
                #Average across trials
                injected_pred_avg_arr = np.mean(np.array(profile_by_task), axis = 0)
    
                injected_pred_avg_df = pd.DataFrame(injected_pred_avg_arr, columns = list(model_info['cov'][task].keys()))
                injected_pred_avg_df['position'] = list(range(injected_pred_avg_df.shape[0]))
                injected_pred_avg_df = injected_pred_avg_df.melt(id_vars = 'position', var_name = 'channel', value_name = 'pred')
                injected_pred_avg_df['model_name'] = fold_prefix    
                injected_pred_avg_df['task'] = task    
                injected_pred_avg_df['motif'] = motif
                injected_pred_avg_df['seq'] = motif_seq
                injected_pred_df = pd.concat([injected_pred_df, injected_pred_avg_df])
                
null_pred_df.to_csv(f'tsv/insilico/marginalized_background_across_solo_models.tsv.gz', sep = '\t', index = False)
injected_pred_df.to_csv(f'tsv/insilico/marginalized_motifs_across_solo_models.tsv.gz', sep = '\t', index = False)

I0000 00:00:1772722428.500362 4032127 service.cc:148] XLA service 0x147a7800a760 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772722428.500410 4032127 service.cc:156]   StreamExecutor device (0): Host, Default Version
2026-03-05 08:53:48.544948: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1772722428.736052 4032127 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


## Plot marginalizations

In [28]:
null_pred_df = pd.read_csv(f'tsv/insilico/marginalized_background_across_solo_models.tsv.gz', sep = '\t')
injected_pred_df = pd.read_csv(f'tsv/insilico/marginalized_motifs_across_solo_models.tsv.gz', sep = '\t')

injected_pred_df[['model', 'fold_name']] = injected_pred_df['model_name'].str.split('_fold', n=1, expand=True)
null_pred_df[['model', 'fold_name']] = null_pred_df['model_name'].str.split('_fold', n=1, expand=True)

injected_pred_df['pred'][injected_pred_df['channel']=='neg'] = injected_pred_df['pred'][injected_pred_df['channel']=='neg'].values * (-1)
null_pred_df['pred'][null_pred_df['channel']=='neg'] = null_pred_df['pred'][null_pred_df['channel']=='neg'].values * (-1)

injected_pred_df.tail()

,position,channel,pred,model_name,task,motif,seq,model,fold_name
95995,995,neg,-0.388122,s2_anthox1a_pbx_ha_nexus_fold3,pbx,Dfd,TTAATGGC,s2_anthox1a_pbx_ha_nexus,3
95996,996,neg,-0.398127,s2_anthox1a_pbx_ha_nexus_fold3,pbx,Dfd,TTAATGGC,s2_anthox1a_pbx_ha_nexus,3
95997,997,neg,-0.410542,s2_anthox1a_pbx_ha_nexus_fold3,pbx,Dfd,TTAATGGC,s2_anthox1a_pbx_ha_nexus,3
95998,998,neg,-0.422820,s2_anthox1a_pbx_ha_nexus_fold3,pbx,Dfd,TTAATGGC,s2_anthox1a_pbx_ha_nexus,3
95999,999,neg,-0.413633,s2_anthox1a_pbx_ha_nexus_fold3,pbx,Dfd,TTAATGGC,s2_anthox1a_pbx_ha_nexus,3


In [29]:
for model_name,model_info in tqdm(modeling_design_dict.items()):
    
    inj_df = injected_pred_df[injected_pred_df.model==model_name]
    null_df = null_pred_df[null_pred_df.model==model_name]
    
    showcase_injections_plot = (ggplot()+
        geom_line(data = null_df,
                  mapping = aes(x = 'position', y = 'pred', group = 'channel'), color = 'gray', alpha = .5, size = 1)+
        geom_line(data = inj_df,
                  mapping = aes(x = 'position', y = 'pred', group = 'channel', color = 'task'), size = 1)+
        scale_x_continuous(name = 'Predicted position')+
        scale_y_continuous(name = 'Predicted accessibility')+
        facet_grid('motif ~ fold_name', scales = 'fixed')+
        ggtitle(model_name)+
        theme_classic() + 
        theme(figure_size=(24, 24)))
    
    showcase_injections_plot.save(f'{figure_path}/marginalizations_solo_{model_name}.png')
    showcase_injections_plot.save(f'{figure_path}/marginalizations_solo_{model_name}.pdf')

100%|█████████████████████████████████████████████| 4/4 [00:14<00:00,  3.67s/it]
